# Cats vs Dogs!

<img src="https://raw.githubusercontent.com/benjum/UCLAX-ML/main/Data/cat-dog-data/test/cat/cat.1500.jpg" width="250px"/> <img src="https://raw.githubusercontent.com/benjum/UCLAX-ML/main/Data/cat-dog-data/test/dog/dog.1500.jpg" width="250px"/>

#### We're going to use PyTorch and Convolutional Neural Networks to identify whether a picture is a dog or a cat.

Side-note: I think the dataset is biased against cats, because most of the cats I've looked at in this dataset look crazy, a bit ugly, or like they totally have it out for the dogs!

This dataset is a subset of the data that can be obtained at [Kaggle's Dogs Vs Cats page](https://www.kaggle.com/c/dogs-vs-cats/data).

The code below was motivated by Chollet's Deep Learning with Python book, in which you will find many, many more interesting details and tidbits about doing deep learning.

In [ ]:
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader, Subset

In [ ]:
torch.__version__

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

Training on these images can take a long time on a CPU, so a GPU is recommended. To check that everything works before a long run, set `QUICK_TEST = True`: each model then trains for 1 epoch on 64 images per set. (The results will be meaningless, but any errors will show up quickly.)

In [ ]:
QUICK_TEST = False
NUM_WORKERS = 4 if device == "cuda" else 0     # background processes that load images during training

## Getting our data

Our data images are in folders that are already split up into training, test, and validation sets. Furthermore, they are already split up into cat and dog folders.

`torchvision.datasets.ImageFolder` gets class labels based on whether the images are retrieved from these "cat" or "dog" folders: each subfolder becomes a class, numbered in alphabetical order (`cat` = 0, `dog` = 1).

Each image is resized to 180×180 pixels and converted to a tensor of floats between 0 and 1 (the pixel values 0–255 divided by 255), with shape `(3, 180, 180)`: 3 color channels, then height and width.

In [ ]:
DATA_DIR = Path("..", "..", "Data", "cat-dog-data")
DATA_DIR.resolve(), DATA_DIR.is_dir()

In [ ]:
IMAGE_SIZE = 180

preprocess = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),     # 0-255 -> 0-1
])

def image_folder(split, transform):
    dataset = torchvision.datasets.ImageFolder(DATA_DIR / split, transform=transform)
    if QUICK_TEST:   # 32 cats (the first images) and 32 dogs (the last images)
        dataset = Subset(dataset, list(range(32)) + list(range(len(dataset) - 32, len(dataset))))
    return dataset

train_dataset = image_folder("train", preprocess)
validation_dataset = image_folder("validation", preprocess)
test_dataset = image_folder("test", preprocess)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)
validation_loader = DataLoader(validation_dataset, batch_size=32, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS)

print(len(train_dataset), "training images,", len(validation_dataset), "validation,", len(test_dataset), "test")

Brief aside: how can you work with these datasets? Each example is a pair `(image, label)`.

In [ ]:
image, label = train_dataset[2]

The image is at index 0:

In [ ]:
image.shape

The label is at index 1:

In [ ]:
label

In [ ]:
image[0, 2, :10]     # a few pixel values: channel 0 (red), row 2, first 10 columns

In [ ]:
raw = torchvision.datasets.ImageFolder(DATA_DIR / "train")    # no transform: the original photos

b = 1002
picture, label = raw[b]
print("The label for image", b, "is", label, f"({raw.classes[label]})")
print("The picture for image", b, "is")
plt.imshow(picture);

## Building our model

Two convolutional layers, each followed by max pooling, then a dense layer with **one output**. The convolutions have no padding, so each one trims a pixel from every edge.

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3), nn.ReLU(),     # 180 -> 178
    nn.MaxPool2d(2),                                # 178 -> 89
    nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(),    # 89 -> 87
    nn.MaxPool2d(2),                                # 87 -> 43
    nn.Flatten(),                                   # 64 x 43 x 43 = 118,336
    nn.Linear(64 * 43 * 43, 1),                     # one output: the logit for "dog"
).to(device)

In [ ]:
print(model)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

Note that since we are doing binary classification, we now use binary cross-entropy rather than the multiclass cross-entropy we used for digits. As in the review notebook, the model outputs a single **logit** (no sigmoid), and `nn.BCEWithLogitsLoss` applies the sigmoid itself. The labels must be floats with shape `(batch, 1)`.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-3)

During our model fitting, we can save the model in a file after each epoch. This allows us to retrieve model information later if needed.
* Key to our needs, we only save the model's parameters into the file (overwriting the previous one) if the current validation loss is **lower** than at any earlier epoch.
* Our saved file thus holds the model corresponding to its best performance on the validation data.

In PyTorch, `model.state_dict()` holds all of the model's parameters, `torch.save` writes it to a file, and `model.load_state_dict(torch.load(...))` loads it back.

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    # Training if an optimizer is given, otherwise evaluation. Returns (loss, accuracy).
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.float().reshape(-1, 1).to(device)
            logits = model(inputs)
            loss = loss_fn(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += ((logits > 0).float() == labels).sum().item()
            n += len(labels)
    return total_loss / n, correct / n


def fit(model, train_loader, validation_loader, loss_fn, optimizer, epochs, checkpoint_path):
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    for epoch in range(epochs):
        loss, accuracy = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_accuracy = run_epoch(model, validation_loader, loss_fn)
        for key, value in zip(history, (loss, accuracy, val_loss, val_accuracy)):
            history[key].append(value)
        if val_loss < best_val_loss:                          # save only the best model so far
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
        print(f"Epoch {epoch + 1}/{epochs}   loss: {loss:.4f}  accuracy: {accuracy:.4f}"
              f"   val_loss: {val_loss:.4f}  val_accuracy: {val_accuracy:.4f}")
    return history

Do the fit!

Note that we include validation now as a part of this fit. Validation is useful for assessing whether the trained model generalizes well to unseen data, and we do it on data that is not the test data because we may want to alter our model hyperparameters before doing our final training and then our final testing on our test data.

In [ ]:
history = fit(model, train_loader, validation_loader, loss_fn, optimizer,
              epochs=1 if QUICK_TEST else 20,
              checkpoint_path="convnet_from_scratch.pt")

## How does the model perform?

* Does it overfit to the training data?
    * Does the accuracy and loss on the validation data match up reasonably with the accuracy and loss on the training data?
* What is the accuracy of classification?
    * Assess the model on the test data (potentially only using the number of epochs for which validation has shown us that overfitting is not occurring)

In [ ]:
def plot_history(history):
    accuracy = history["accuracy"]
    val_accuracy = history["val_accuracy"]
    loss = history["loss"]
    val_loss = history["val_loss"]

    epochs = range(1, len(accuracy) + 1)

    plt.plot(epochs, accuracy, "bo", label="Training accuracy")
    plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
    plt.title("Training and validation accuracy")
    plt.legend()

    plt.figure()

    plt.plot(epochs, loss, "bo", label="Training loss")
    plt.plot(epochs, val_loss, "b", label="Validation loss")
    plt.title("Training and validation loss")
    plt.legend()

    plt.show()

plot_history(history)

To test the model accuracy on test data, we take advantage of the fact that we've saved the model (in "convnet_from_scratch.pt") at that point for which the validation loss was lowest.
* We can assess the model from a point before we started overfitting
* We can do it without having to retrain the model completely using a smaller number of epochs.

In [ ]:
model.load_state_dict(torch.load("convnet_from_scratch.pt", map_location=device))
test_loss, test_acc = run_epoch(model, test_loader, loss_fn)
print(f"Test accuracy: {test_acc:.3f}")

## Model #2

The overfitting occurs in part because we have such a small dataset. To bypass this without getting a lot of new data, we're going to use a clever bit of data manipulation.

Rather than using new images, we're going to use the same images with a small bit of zoom, rotation, and/or horizontal flipping. This allows us to get images that have different pixel values and that are new images to the algorithm, but that are essentially the same dog or cat image.

In PyTorch, the augmentation is part of the **training** images' transform, so each time an image is loaded it's modified differently. The validation and test images keep the plain `preprocess` transform.
* `RandomHorizontalFlip`: mirror the image half of the time
* `RandomRotation(36)`: rotate by up to ±36° (a tenth of a full turn)
* `RandomAffine(..., scale=(0.8, 1.2))`: zoom in or out by up to 20%

In [ ]:
data_augmentation = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    T.RandomHorizontalFlip(),
    T.RandomRotation(36),
    T.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

In [ ]:
picture = raw[2][0]
plt.figure(figsize=(10, 10))
for i in range(9):
    augmented_image = data_augmentation(picture)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_image.permute(1, 2, 0))     # (channels, height, width) -> (height, width, channels)
    plt.axis("off")

The same model as before, plus a **dropout** layer before the dense layer, which randomly switches off half of its inputs during training.

In [ ]:
augmented_train_loader = DataLoader(image_folder("train", data_augmentation), batch_size=32,
                                    shuffle=True, num_workers=NUM_WORKERS)

torch.manual_seed(42)
model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Dropout(0.5),
    nn.Linear(64 * 43 * 43, 1),
).to(device)

In [ ]:
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-3)

In [ ]:
history = fit(model, augmented_train_loader, validation_loader, loss_fn, optimizer,
              epochs=1 if QUICK_TEST else 60,
              checkpoint_path="convnet_from_scratch_with_augmentation.pt")

In [ ]:
plot_history(history)

This should show much better behavior relative to overfitting!

In [ ]:
model.load_state_dict(torch.load("convnet_from_scratch_with_augmentation.pt", map_location=device))
test_loss, test_acc = run_epoch(model, test_loader, loss_fn)
print(f"Test accuracy: {test_acc:.3f}")

Getting better at accuracy.

We can actually do much better with more convolutional and pooling layers! (although the computational demand for resources will be even higher -- these networks can quickly consume resources!!)

## Checking out a couple images and their cat/dog classification

Each dataset returns one `(image, label)` pair at a time, so we can retrieve example images and feed them into the model for a prediction. The model expects a **batch**, so a single image needs an extra dimension at the front: `image.unsqueeze(0)` has shape `(1, 3, 180, 180)`. The model outputs a logit, and `torch.sigmoid` turns it into the probability of "dog".

In [ ]:
image, label = test_dataset[2]
image.shape

In [ ]:
b = 5
image, label = test_dataset[b]
print("The label for image", b, "is", label)
print("The picture for image", b, "is")
plt.imshow(image.permute(1, 2, 0));

In [ ]:
model.eval()
with torch.no_grad():
    probability_dog = torch.sigmoid(model(test_dataset[5][0].unsqueeze(0).to(device))).item()
probability_dog

In [ ]:
b = np.random.choice(len(test_dataset))
image, label = test_dataset[b]
print("The label for image", b, "is", label)
plt.imshow(image.permute(1, 2, 0))
with torch.no_grad():
    logit = model(image.unsqueeze(0).to(device)).item()
if logit > 0:
    print("Predict Dog")
else:
    print("Predict Cat")